# Music Annotation Project V9.0B
## Notebook 60R1: Fresh SAFE_REMOVE inference on reviewed HITL masks

**Runtime:** GPU. In Google Colab select a T4 GPU, then choose **Runtime → Run all**.

### Purpose
Operationalise the frozen V8.8 REMOVE-only policy on the real Notebook 06 → Notebook 07 reviewed HITL population.

### Page-specific runtime inputs
- Notebook 07 aligned observed page.
- Notebook 07 reviewed/corrected authoritative mask.
- Notebook 06 draft/confidence lineage for audit only.

### Frozen policy inputs
- One complete Notebook 52 V8.8A freeze/replay run.
- The exact model/checkpoint, calibrator, ordered channel schema, thresholds, connected-run rules and compatibility identity named by that run.
- Canonical V7.8 mask preparation, observable channels and baseline reconstruction.

### Non-negotiable rules
- No Notebook 45, 46 or 50 **page output** is required for a new HITL page.
- Notebook 50 may be referenced only as the recorded producer of frozen learned objects named by Notebook 52.
- No clean page is loaded.
- No training, fitting, recalibration, threshold change or policy selection occurs.
- SAFE_PROTECT is prohibited.
- `M5_REMOVE = M3 & ~SAFE_REMOVE`.
- Unsupported, uncertain, invalid and failed coordinates retain exact M3 treatment.
- No pixel outside the authoritative prepared mask may change.
- The notebook stops if the frozen bundle cannot unambiguously describe fresh-page inference. It never invents an ensemble rule or selects a convenient model file.

In [9]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow torch torchvision scikit-learn joblib matplotlib

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Configuration and run identity

In [10]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
os.environ['PYTHONHASHSEED']='42'

from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, random, sys, platform, gc, time, shutil, traceback, zipfile
import cv2, torch, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch import nn

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
DRAFT_AUDIT_DIR=ROOT/'human_in_the_middle'/'draft_audit'
REVIEW_AUDIT_DIR=ROOT/'human_in_the_middle'/'audit'
N52_ROOT=ROOT/'m5_remove_policy_freeze_v8_8a'
OUTPUT_ROOT=ROOT/'v9_0b_notebook60r1_fresh_safe_remove'

NOTEBOOK52_RUN_OVERRIDE=None
PAGE_ALLOWLIST=None
EXPECTED_REVIEWED_PAGES=151
REQUIRE_EXACT_151=False
BATCH_SIZE=1024
STOP_AT_FIRST_PAGE_FAILURE=False
MAX_VISUAL_PAGES=30
SEED=42
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE!='cuda': raise RuntimeError('Notebook 60R1 requires a GPU runtime for fresh frozen CNN inference.')
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic=True
torch.backends.cudnn.benchmark=False

RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook60r1_v9_0b'
OUT=OUTPUT_ROOT/RUN
INPUTS=OUT/'inputs'; PAGES=OUT/'pages'; TABLES=OUT/'tables'; AUDIT=OUT/'audit'; FAILURES=OUT/'failures'; VISUAL=OUT/'visual'; UPLOAD=OUT/'upload'
for d in [OUT,INPUTS,PAGES,TABLES,AUDIT,FAILURES,VISUAL,UPLOAD]: d.mkdir(parents=True,exist_ok=False)
print('GPU:',torch.cuda.get_device_name(0))
print('Output:',OUT)

GPU: Tesla T4
Output: /content/drive/MyDrive/MusicAnnotationProject/v9_0b_notebook60r1_fresh_safe_remove/20261111_090501_notebook60r1_v9_0b


## 2. General helpers

In [11]:
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}

def require(x,msg):
    if not x: raise RuntimeError(msg)

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()

def object_hash(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()

def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()

def atomic_json(path,value):
    path=Path(path); tmp=Path(str(path)+'.tmp'); tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str)); tmp.replace(path)

def read_json(path): return json.loads(Path(path).read_text())

def read_gray(path):
    a=cv2.imread(str(path),cv2.IMREAD_GRAYSCALE)
    require(a is not None,f'Unreadable image: {path}')
    return a

def latest_complete(roots,required_any):
    hits=[]
    for root in roots:
        if not Path(root).exists(): continue
        for p in Path(root).rglob('*'):
            if not p.is_dir(): continue
            if all(any((p/name).exists() for name in choices) for choices in required_any): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

def md5_file(path):
    h=hashlib.md5()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()

def binary(a): return (np.asarray(a)>0)

def reset_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
reset_seed()

## 3. Resolve the frozen Notebook 52 policy bundle

This resolver is manifest-led. It does not select an arbitrary `.pt` or `.joblib`. It first resolves one complete Notebook 52 run, then reads its policy and artifact records. All paths and hashes must be named by those records.

In [12]:
N52_REQUIRED=[
    'notebook52_verdict.json',
    'policy_bundle/policy_manifest.json',
    'replay/source_fingerprint.json',
    'tables/notebook50_full_artifact_inventory.csv',
    'tables/binary_artifact_loadability.csv',
    'tables/m5_remove_golden_master_semantic_hashes.csv'
]
roots=[N52_ROOT]
N52=Path(NOTEBOOK52_RUN_OVERRIDE) if NOTEBOOK52_RUN_OVERRIDE else latest_complete(roots,N52_REQUIRED)
require(N52 is not None,'No complete Notebook 52 V8.8A freeze/replay run was resolved.')

def first_existing(folder,names):
    hits=[folder/name for name in names if (folder/name).exists()]
    require(len(hits)==1,f'Expected exactly one of {names} in {folder}; found {hits}')
    return hits[0]

POLICY_PATH=first_existing(N52,N52_REQUIRED[0])
INVENTORY_PATH=first_existing(N52,N52_REQUIRED[1])
REPLAY_PATH=first_existing(N52,N52_REQUIRED[2])
policy=read_json(POLICY_PATH)
replay=read_json(REPLAY_PATH)
inv=pd.read_csv(INVENTORY_PATH)

require(bool(replay.get('passed',replay.get('replay_passed',False))),'Notebook 52 replay record is not PASS.')
require('SAFE_PROTECT' not in json.dumps(policy).upper() or policy.get('safe_protect_runtime_enabled') is False,'Policy appears to authorise SAFE_PROTECT.')

# Normalize inventory columns without weakening identity.
colmap={c.lower().strip():c for c in inv.columns}
path_col=next((colmap[x] for x in ['relative_path','path','file','artifact_path','artefact_path'] if x in colmap),None)
hash_col=next((colmap[x] for x in ['sha256','file_sha256','hash'] if x in colmap),None)
role_col=next((colmap[x] for x in ['role','semantic_role','artifact_role','artefact_role'] if x in colmap),None)
require(path_col and hash_col and role_col,f'Notebook 52 inventory lacks path/hash/role columns: {list(inv.columns)}')

def resolve_inventory_row(row):
    raw=Path(str(row[path_col]))
    candidates=[raw,N52/raw]
    source_run=policy.get('source_notebook50_run') or policy.get('authoritative_notebook50_run') or policy.get('source_run')
    if source_run: candidates.append(Path(source_run)/raw)
    existing=[p for p in candidates if p.exists()]
    require(len(existing)==1,f'Ambiguous or missing frozen artifact {raw}: {existing}')
    p=existing[0]
    require(sha256_file(p)==str(row[hash_col]),f'Frozen artifact hash mismatch: {p}')
    return p

resolved=[]
for _,r in inv.iterrows():
    try:
        p=resolve_inventory_row(r)
        resolved.append({'role':str(r[role_col]),'path':str(p),'sha256':str(r[hash_col]),'bytes':p.stat().st_size})
    except Exception as e:
        resolved.append({'role':str(r[role_col]),'path':str(r[path_col]),'sha256':str(r[hash_col]),'error':str(e)})
resolved_df=pd.DataFrame(resolved)
resolved_df.to_csv(AUDIT/'resolved_frozen_artifacts.csv',index=False)
require('error' not in resolved_df.columns or resolved_df.error.isna().all(),'One or more frozen artifacts failed resolution. See resolved_frozen_artifacts.csv')

atomic_json(AUDIT/'notebook52_source.json',{
    'notebook52_run':str(N52),'policy_path':str(POLICY_PATH),'policy_sha256':sha256_file(POLICY_PATH),
    'inventory_path':str(INVENTORY_PATH),'inventory_sha256':sha256_file(INVENTORY_PATH),
    'replay_path':str(REPLAY_PATH),'replay_sha256':sha256_file(REPLAY_PATH),
    'policy_id':policy.get('policy_id'),'compatibility_id':policy.get('compatibility_id')
})
print('Notebook 52:',N52)
print('Policy ID:',policy.get('policy_id'))
print(resolved_df[['role','path']].to_string(index=False))

RuntimeError: Expected exactly one of policy_bundle/policy_manifest.json in /content/drive/MyDrive/MusicAnnotationProject/m5_remove_policy_freeze_v8_8a/20261102_123709_notebook52_v8_8a; found [PosixPath('/'), PosixPath('/content/drive/MyDrive/MusicAnnotationProject/m5_remove_policy_freeze_v8_8a/20261102_123709_notebook52_v8_8a')]

## 4. Interpret the exact frozen fresh-inference contract

In [ ]:
def rows_for_role(patterns):
    role=resolved_df.role.astype(str).str.lower()
    m=np.logical_or.reduce([role.str.contains(p.lower(),regex=False).to_numpy() for p in patterns])
    return resolved_df.loc[m].copy()

schema_rows=rows_for_role(['channel schema','predictor schema','schema'])
policy_rows=rows_for_role(['policy','threshold','decision'])
model_rows=rows_for_role(['model','checkpoint'])
cal_rows=rows_for_role(['calibrator','isotonic'])
require(len(schema_rows)>=1,'Frozen inventory does not identify a channel/predictor schema.')
require(len(model_rows)>=1,'Frozen inventory does not identify a model/checkpoint.')
require(len(cal_rows)>=1,'Frozen inventory does not identify a calibrator.')

# Prefer explicit policy references. Otherwise require unique role candidate.
def policy_named_path(keys, candidates, label):
    names=[policy.get(k) for k in keys if policy.get(k)]
    if names:
        target=Path(str(names[0])).name
        hits=candidates[candidates.path.map(lambda x:Path(x).name==target)]
        require(len(hits)==1,f'Policy-named {label} {target} does not resolve uniquely.')
        return Path(hits.iloc[0].path)
    require(len(candidates)==1,f'Frozen bundle has {len(candidates)} {label} candidates and policy does not name one.')
    return Path(candidates.iloc[0].path)

SCHEMA_PATH=policy_named_path(['channel_schema','channel_schema_path','predictor_schema','schema_path'],schema_rows,'schema')
schema=read_json(SCHEMA_PATH)
channels=schema.get('channels') or schema.get('ordered_channels') or schema.get('ordered_predictors')
require(isinstance(channels,list) and channels,'Frozen schema contains no ordered channels.')
PATCH=int(schema.get('patch',schema.get('patch_size',33)))
require(PATCH%2==1,'Patch size must be odd.')
HALF=PATCH//2

PRINT_THRESHOLD=float(policy.get('print_threshold',policy.get('threshold',0.995)))
ANNOTATION_THRESHOLD=float(policy.get('annotation_threshold',0.995))
PRINT_MARGIN=float(policy.get('print_margin',policy.get('decision_margin',0.990)))
MIN_PRINT_RUN=int(policy.get('minimum_print_run',policy.get('min_print_run',2)))
DARK_THRESHOLD=int(policy.get('dark_threshold',200))
MASK_DILATION=int(policy.get('mask_dilation',2))
CROP_MARGIN=int(policy.get('crop_margin',48))
require(PRINT_THRESHOLD==0.995 and ANNOTATION_THRESHOLD==0.995 and PRINT_MARGIN==0.990,'Frozen decision constants differ from the registered V8.8 contract; inspect policy rather than silently proceeding.')
require(MIN_PRINT_RUN>=2,'Frozen policy permits singleton SAFE_REMOVE; blocked.')

# Fresh inference can be either a single all-development model or an explicitly declared ensemble.
aggregation=policy.get('model_aggregation') or policy.get('fresh_inference_aggregation') or policy.get('ensemble_aggregation')
selected_names=policy.get('selected_models') or policy.get('model_files') or policy.get('checkpoints')
if isinstance(selected_names,str): selected_names=[selected_names]
if selected_names:
    selected=[]
    for name in selected_names:
        hits=model_rows[model_rows.path.map(lambda x:Path(x).name==Path(str(name)).name)]
        require(len(hits)==1,f'Selected model {name} not uniquely resolved.')
        selected.append(Path(hits.iloc[0].path))
    MODEL_PATHS=selected
else:
    require(len(model_rows)==1,'Multiple frozen models exist but no selected_models/aggregation contract is recorded.')
    MODEL_PATHS=[Path(model_rows.iloc[0].path)]
if len(MODEL_PATHS)>1:
    require(aggregation in ['mean_probability','mean_raw_probability'],'Multiple models require an explicit mean-probability aggregation contract.')

selected_cals=policy.get('selected_calibrators') or policy.get('calibrator_files')
if isinstance(selected_cals,str): selected_cals=[selected_cals]
if selected_cals:
    CALIBRATOR_PATHS=[]
    for name in selected_cals:
        hits=cal_rows[cal_rows.path.map(lambda x:Path(x).name==Path(str(name)).name)]
        require(len(hits)==1,f'Selected calibrator {name} not uniquely resolved.')
        CALIBRATOR_PATHS.append(Path(hits.iloc[0].path))
else:
    require(len(cal_rows)==len(MODEL_PATHS) or len(cal_rows)==1,'Calibrator count does not match the fresh inference contract.')
    CALIBRATOR_PATHS=[Path(x) for x in cal_rows.path]
if len(CALIBRATOR_PATHS)==1 and len(MODEL_PATHS)>1: CALIBRATOR_PATHS=CALIBRATOR_PATHS*len(MODEL_PATHS)
require(len(CALIBRATOR_PATHS)==len(MODEL_PATHS),'Model/calibrator pairing is incomplete.')

contract={'policy_id':policy.get('policy_id'),'compatibility_id':policy.get('compatibility_id'),'channels':channels,'patch':PATCH,'model_paths':[str(x) for x in MODEL_PATHS],'calibrator_paths':[str(x) for x in CALIBRATOR_PATHS],'model_aggregation':aggregation,'print_threshold':PRINT_THRESHOLD,'annotation_threshold':ANNOTATION_THRESHOLD,'print_margin':PRINT_MARGIN,'minimum_print_run':MIN_PRINT_RUN,'dark_threshold':DARK_THRESHOLD,'mask_dilation':MASK_DILATION}
atomic_json(AUDIT/'fresh_inference_contract.json',contract)
print(json.dumps(contract,indent=2))

## 5. Canonical V7.8 page preparation and observable channels

In [ ]:
def immutable(original,candidate,mask):
    out=original.copy(); out[mask>0]=candidate[mask>0]
    require(np.array_equal(out[mask==0],original[mask==0]),'Outside-mask immutability failure.')
    return out

def prepare_mask(seed,dilation=MASK_DILATION):
    m=np.where(seed>0,255,0).astype(np.uint8)
    if dilation>0: m=cv2.dilate(m,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*dilation+1,2*dilation+1)))
    return m

def padded_components(mask):
    n,lab,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8); h,w=mask.shape; boxes=[]
    for i in range(1,n):
        x,y,bw,bh,a=stats[i]
        if a>=8: boxes.append((max(0,x-CROP_MARGIN),max(0,y-CROP_MARGIN),min(w,x+bw+CROP_MARGIN),min(h,y+bh+CROP_MARGIN)))
    changed=True
    while changed:
        changed=False; merged=[]
        while boxes:
            ax0,ay0,ax1,ay1=boxes.pop(); rest=[]
            for bx0,by0,bx1,by1 in boxes:
                if not (ax1<bx0 or bx1<ax0 or ay1<by0 or by1<ay0):
                    ax0=min(ax0,bx0); ay0=min(ay0,by0); ax1=max(ax1,bx1); ay1=max(ay1,by1); changed=True
                else: rest.append((bx0,by0,bx1,by1))
            boxes=rest; merged.append((ax0,ay0,ax1,ay1))
        boxes=merged
    return boxes

def telea_reconstruct(observed,mask):
    result=observed.copy()
    for x0,y0,x1,y1 in padded_components(mask):
        im=observed[y0:y1,x0:x1]; mk=mask[y0:y1,x0:x1]
        cand=cv2.inpaint(im,mk,3,cv2.INPAINT_TELEA); local=result[y0:y1,x0:x1]; local[mk>0]=cand[mk>0]
    return immutable(observed,result,mask)

def background_surface(observed,mask,sigma=45.0):
    visible=(mask==0)&(observed>=200); values=observed.astype(np.float32)*visible; weights=visible.astype(np.float32)
    num=cv2.GaussianBlur(values,(0,0),sigma); den=cv2.GaussianBlur(weights,(0,0),sigma); surface=num/np.maximum(den,1e-4); fallback=cv2.GaussianBlur(observed,(0,0),sigma)
    return np.clip(np.where(den>0.02,surface,fallback),0,255).astype(np.uint8)

def support_maps(observed,mask,band=4):
    dark=(observed<190)&(mask==0)
    h=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0
    v=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((15,1),np.uint8))>0
    n,lab,stats,_=cv2.connectedComponentsWithStats(dark.astype(np.uint8),8); c=np.zeros_like(dark)
    for i in range(1,n):
        x,y,w,hg,a=stats[i]
        if 3<=a<=650 and w<=40 and hg<=40: c[lab==i]=True
    k=np.ones((2*band+1,2*band+1),np.uint8)
    return cv2.dilate(h.astype(np.uint8),k)>0,cv2.dilate(v.astype(np.uint8),k)>0,cv2.dilate(c.astype(np.uint8),k)>0

def geometry_t80(observed,mask,telea):
    paper=background_surface(observed,mask); norm=cv2.divide(observed,np.maximum(paper,1),scale=255); horizontal,vertical,compact=support_maps(observed,mask); m=mask>0
    preserve=m&(horizontal|vertical|compact)&(observed<200); remove=m&~preserve&(observed>=80)&(norm>=80); ambiguous=m&~preserve&~remove
    result=observed.copy(); result[remove]=paper[remove]; result[ambiguous]=telea[ambiguous]; result[preserve]=observed[preserve]
    return immutable(observed,result,mask),{'preserve':preserve,'remove':remove,'ambiguous':ambiguous,'horizontal':horizontal,'vertical':vertical,'compact':compact,'paper':paper,'norm':norm}

def page_channels(observed,mask,routes):
    arrays={
        'obs':observed.astype(np.float32)/255.0,
        'norm':routes['norm'].astype(np.float32)/255.0,
        'mask':(mask>0).astype(np.float32),
        'depth':cv2.distanceTransform((mask>0).astype(np.uint8),cv2.DIST_L2,5).astype(np.float32),
        'horizontal':routes['horizontal'].astype(np.float32),'vertical':routes['vertical'].astype(np.float32),'compact':routes['compact'].astype(np.float32),
        'preserve':routes['preserve'].astype(np.float32),'remove':routes['remove'].astype(np.float32),'ambiguous':routes['ambiguous'].astype(np.float32),
        'edge':(cv2.Canny(observed,70,170)>0).astype(np.float32),
    }
    aliases={'observed':'obs','normalised':'norm','normalized':'norm','authoritative_mask':'mask','mask_distance':'depth','edges':'edge'}
    ordered=[]
    for name in channels:
        key=aliases.get(name,name)
        require(key in arrays,f'Frozen channel {name} is not provided by the canonical builder.')
        ordered.append((name,arrays[key]))
    return ordered

## 6. Frozen CNN model and calibrated inference

In [ ]:
class TopologyCNN(nn.Module):
    def __init__(self,nc):
        super().__init__()
        self.net=nn.Sequential(nn.Conv2d(nc,24,3,padding=1),nn.ReLU(),nn.BatchNorm2d(24),nn.MaxPool2d(2),nn.Conv2d(24,48,3,padding=1),nn.ReLU(),nn.BatchNorm2d(48),nn.MaxPool2d(2),nn.Conv2d(48,64,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1))
        self.head=nn.Linear(64,2)
    def forward(self,x): return self.head(self.net(x).flatten(1))

def load_checkpoint(path):
    raw=torch.load(path,map_location=DEVICE)
    state=raw.get('state_dict',raw.get('model_state_dict',raw)) if isinstance(raw,dict) else raw
    # strip common wrapper prefix only; architecture mismatch remains a hard failure.
    state={str(k).removeprefix('module.'):v for k,v in state.items()}
    m=TopologyCNN(len(channels)).to(DEVICE); m.load_state_dict(state,strict=True); m.eval(); return m

models=[load_checkpoint(p) for p in MODEL_PATHS]
calibrators=[joblib.load(p) for p in CALIBRATOR_PATHS]

def score_coordinates(channel_arrays,ys,xs):
    pads=[np.pad(a,HALF,mode='reflect') for _,a in channel_arrays]
    member=[]
    for model,cal in zip(models,calibrators):
        scores=[]
        with torch.no_grad():
            for start in range(0,len(ys),BATCH_SIZE):
                yy=ys[start:start+BATCH_SIZE]; xx=xs[start:start+BATCH_SIZE]
                batch=np.stack([np.stack([p[y:y+PATCH,x:x+PATCH] for p in pads]) for y,x in zip(yy,xx)]).astype(np.float32)
                prob=torch.softmax(model(torch.from_numpy(batch).to(DEVICE)),1)[:,1].cpu().numpy().astype(np.float64)
                scores.append(np.asarray(cal.predict(prob),dtype=np.float64))
        member.append(np.concatenate(scores) if scores else np.empty(0,float))
    mat=np.vstack(member)
    if len(member)==1: return mat[0],mat
    require(aggregation in ['mean_probability','mean_raw_probability'],'Unregistered aggregation rule.')
    return mat.mean(axis=0),mat

def decisions_from_probability(p):
    p=np.asarray(p,float); q=1-p; d=np.full(len(p),'UNCERTAIN',object)
    d[q>=ANNOTATION_THRESHOLD]='ANNOTATION'
    d[(p>=PRINT_THRESHOLD)&((p-q)>=PRINT_MARGIN)]='PRINT'
    return d

def enforce_run_rule(shape,ys,xs,decisions,probabilities):
    out=np.asarray(decisions,object).copy(); pm=np.zeros(shape,np.uint8); hit=np.where(out=='PRINT')[0]; pm[ys[hit],xs[hit]]=1
    n,lab,stats,_=cv2.connectedComponentsWithStats(pm,8)
    for cid in range(1,n):
        coords=np.where(lab[ys,xs]==cid)[0]
        if len(coords)<MIN_PRINT_RUN or float(np.mean(probabilities[coords]))<PRINT_THRESHOLD or np.any(probabilities[coords]<PRINT_THRESHOLD): out[coords]='UNCERTAIN'
    return out
print('Frozen inference objects loaded:',len(models),'model(s)')

## 7. Resolve the Notebook 06 → 07 reviewed cohort

In [ ]:
def read_audits(folder,label):
    records=[]; failures=[]
    for p in sorted(Path(folder).glob('*.json')):
        try:
            d=read_json(p); require(isinstance(d,dict),'root is not object'); d['_audit_path']=str(p); d['_type']=label; records.append(d)
        except Exception as e: failures.append({'type':label,'path':str(p),'error':str(e)})
    return records,failures

reviews,rf=read_audits(REVIEW_AUDIT_DIR,'review'); drafts,df=read_audits(DRAFT_AUDIT_DIR,'draft')
pd.DataFrame(rf+df).to_csv(AUDIT/'audit_parse_failures.csv',index=False)
require(not (rf+df),'Unreadable HITL audit records exist.')

def path_from(record,keys,required=True):
    for k in keys:
        v=record.get(k)
        if v and Path(v).exists(): return Path(v),k
    if required: raise RuntimeError(f"{record.get('_audit_path')}: no existing path for {keys}")
    return None,None

drafts_by_md5={}
for d in drafts:
    h=str(d.get('source_md5','')).lower()
    if h: drafts_by_md5.setdefault(h,[]).append(d)

rows=[]; unresolved=[]
for r in reviews:
    try:
        observed,ok=path_from(r,['aligned_output','aligned_page','observed_output'])
        reviewed,mk=path_from(r,['corrected_mask_output','corrected_mask','mask_output'])
        src,_=path_from(r,['source','source_file'],required=False)
        h=str(r.get('source_md5','')).lower()
        candidates=drafts_by_md5.get(h,[])
        draft=candidates[0] if len(candidates)==1 else None
        draft_mask=confidence=None; draft_audit=None
        if draft:
            draft_mask,_=path_from(draft,['draft_mask','draft_mask_output'],required=False)
            confidence,_=path_from(draft,['confidence_map','confidence_map_output'],required=False)
            draft_audit=draft['_audit_path']
        obs=read_gray(observed); seed=read_gray(reviewed); require(obs.shape==seed.shape,'observed/reviewed shape mismatch')
        page=Path(observed).stem
        rows.append({'page':page,'observed_path':str(observed),'reviewed_mask_path':str(reviewed),'source_path':str(src) if src else None,'source_md5':h or None,'review_audit_path':r['_audit_path'],'draft_audit_path':draft_audit,'draft_mask_path':str(draft_mask) if draft_mask else None,'confidence_map_path':str(confidence) if confidence else None,'height':obs.shape[0],'width':obs.shape[1]})
    except Exception as e: unresolved.append({'review_audit_path':r.get('_audit_path'),'error':str(e)})
roster=pd.DataFrame(rows)
if PAGE_ALLOWLIST is not None: roster=roster[roster.page.isin(set(PAGE_ALLOWLIST))].copy()
require(not roster.empty,'No reviewed HITL pages resolved.')
dup=roster[roster.duplicated('page',keep=False)]
require(dup.empty,f'Duplicate page identities: {dup.page.tolist()}')
roster=roster.sort_values('page').reset_index(drop=True)
roster.to_csv(TABLES/'hitl_reviewed_roster.csv',index=False); pd.DataFrame(unresolved).to_csv(TABLES/'unresolved_review_lineages.csv',index=False)
if REQUIRE_EXACT_151: require(len(roster)==EXPECTED_REVIEWED_PAGES,f'Expected 151 pages, resolved {len(roster)}')
print('Resolved reviewed pages:',len(roster),'reference expectation:',EXPECTED_REVIEWED_PAGES)

## 8. Fresh SAFE_REMOVE batch inference and M5 reconstruction

In [ ]:
page_rows=[]; pixel_parts=[]; failures=[]
for pos,r in roster.iterrows():
    page=str(r.page); d=PAGES/page; t0=time.time()
    try:
        observed=read_gray(r.observed_path); seed=read_gray(r.reviewed_mask_path); require(observed.shape==seed.shape,'shape mismatch')
        authority=prepare_mask(seed,MASK_DILATION); telea=telea_reconstruct(observed,authority); baseline,routes=geometry_t80(observed,authority,telea)
        m3=authority>0
        eligible=m3&(observed<DARK_THRESHOLD)&~routes['preserve']
        ys,xs=np.where(eligible)
        ordered=page_channels(observed,authority,routes)
        calibrated,members=score_coordinates(ordered,ys,xs)
        require(np.isfinite(calibrated).all(),'non-finite calibrated probabilities')
        pre=decisions_from_probability(calibrated); final=enforce_run_rule(observed.shape,ys,xs,pre,calibrated)
        safe_remove=np.zeros(observed.shape,bool); safe_remove[ys[final=='PRINT'],xs[final=='PRINT']]=True
        require(np.all(~safe_remove|m3),'SAFE_REMOVE outside M3')
        m5=m3&~safe_remove
        # V8.8 REMOVE-only semantics: M5 is the remaining correction mask. SAFE_REMOVE locations retain exact observed bytes.
        corrected=telea_reconstruct(observed,m5.astype(np.uint8)*255)
        corrected[safe_remove]=observed[safe_remove]
        require(np.array_equal(corrected[~m3],observed[~m3]),'outside-authority drift')
        require(np.array_equal(m5,m3&~safe_remove),'M5 formula mismatch')
        d.mkdir(parents=True,exist_ok=False)
        cv2.imwrite(str(d/'observed.png'),observed); cv2.imwrite(str(d/'authoritative_seed.png'),(seed>0).astype(np.uint8)*255); cv2.imwrite(str(d/'m3.png'),m3.astype(np.uint8)*255); cv2.imwrite(str(d/'safe_remove.png'),safe_remove.astype(np.uint8)*255); cv2.imwrite(str(d/'m5_remove.png'),m5.astype(np.uint8)*255); cv2.imwrite(str(d/'v7_8_baseline.png'),baseline); cv2.imwrite(str(d/'corrected.png'),corrected)
        prob_img=np.zeros(observed.shape,np.float32); prob_img[ys,xs]=calibrated; dec=np.zeros(observed.shape,np.uint8); dec[ys[final=='ANNOTATION'],xs[final=='ANNOTATION']]=85; dec[ys[final=='UNCERTAIN'],xs[final=='UNCERTAIN']]=170; dec[ys[final=='PRINT'],xs[final=='PRINT']]=255
        np.savez_compressed(d/'fresh_inference.npz',eligible=eligible.astype(np.uint8),p_safe_remove=prob_img,decision_map=dec,m3=m3.astype(np.uint8),safe_remove=safe_remove.astype(np.uint8),m5_remove=m5.astype(np.uint8))
        cv2.imwrite(str(d/'decision_map.png'),dec); diff=np.zeros_like(observed); diff[corrected!=baseline]=255; cv2.imwrite(str(d/'difference_from_v78.png'),diff)
        pr=pd.DataFrame({'page':page,'y':ys,'x':xs,'p_safe_remove':calibrated,'decision_pre_run':pre,'decision':final})
        for i in range(members.shape[0]): pr[f'member_{i+1}_p']=members[i]
        part=TABLES/f'{page}_fresh_safe_remove.parquet'; pr.to_parquet(part,index=False); pixel_parts.append(part)
        row={'page':page,'status':'PASS','eligible_pixels':len(pr),'safe_remove_pixels':int(safe_remove.sum()),'annotation_pixels':int((final=='ANNOTATION').sum()),'uncertain_pixels':int((final=='UNCERTAIN').sum()),'m3_pixels':int(m3.sum()),'m5_pixels':int(m5.sum()),'outside_authority_drift':int(np.sum(corrected[~m3]!=observed[~m3])),'m5_formula_xor':int(np.logical_xor(m5,m3&~safe_remove).sum()),'seconds':time.time()-t0,'observed_sha256':array_hash(observed),'m3_sha256':array_hash(m3.astype(np.uint8)),'safe_remove_sha256':array_hash(safe_remove.astype(np.uint8)),'m5_sha256':array_hash(m5.astype(np.uint8)),'corrected_sha256':array_hash(corrected)}
        atomic_json(d/'page_manifest.json',row); page_rows.append(row); print(f'[{pos+1}/{len(roster)}] PASS {page}: SAFE_REMOVE={row["safe_remove_pixels"]:,}')
    except Exception as e:
        failure={'page':page,'status':'FAIL','error':str(e),'traceback':traceback.format_exc(),'review_audit_path':r.review_audit_path}; atomic_json(FAILURES/f'{page}.json',failure); failures.append(failure); print(f'[{pos+1}/{len(roster)}] FAIL {page}: {e}')
        if STOP_AT_FIRST_PAGE_FAILURE: raise

page_df=pd.DataFrame(page_rows); fail_df=pd.DataFrame(failures)
page_df.to_csv(TABLES/'page_results.csv',index=False); fail_df.to_csv(TABLES/'page_failures.csv',index=False)
if pixel_parts:
    # Durable combined table after page-level bounded-memory inference.
    pd.concat([pd.read_parquet(p) for p in pixel_parts],ignore_index=True).to_parquet(TABLES/'all_fresh_safe_remove_predictions.parquet',index=False)
print('Passed:',len(page_df),'Failed:',len(fail_df))

## 9. Deterministic replay and operational integrity verdict

In [ ]:
require(len(page_df)>0,'No page completed successfully.')
# Replay the first successful page from persisted probabilities/decisions and verify M5/output identity.
sample=page_df.iloc[0].page; rr=roster.set_index('page').loc[sample]; observed=read_gray(rr.observed_path); seed=read_gray(rr.reviewed_mask_path); m3=read_gray(PAGES/sample/'m3.png')>0; safe=read_gray(PAGES/sample/'safe_remove.png')>0; m5=read_gray(PAGES/sample/'m5_remove.png')>0; saved=read_gray(PAGES/sample/'corrected.png')
replayed=telea_reconstruct(observed,m5.astype(np.uint8)*255); replayed[safe]=observed[safe]
replay_record={'page':sample,'m5_formula_exact':bool(np.array_equal(m5,m3&~safe)),'corrected_exact':bool(np.array_equal(saved,replayed)),'saved_sha256':array_hash(saved),'replayed_sha256':array_hash(replayed)}
atomic_json(AUDIT/'deterministic_replay.json',replay_record)

checks={
    'notebook52_replay_passed':True,
    'frozen_artifact_hashes_passed':True,
    'fresh_inference_contract_unambiguous':True,
    'all_resolved_pages_accounted_for':len(page_df)+len(fail_df)==len(roster),
    'zero_page_failures':len(fail_df)==0,
    'zero_outside_authority_drift':bool((page_df.outside_authority_drift==0).all()),
    'zero_m5_formula_xor':bool((page_df.m5_formula_xor==0).all()),
    'deterministic_replay_exact':bool(replay_record['m5_formula_exact'] and replay_record['corrected_exact']),
    'no_clean_truth_loaded':True,
    'safe_protect_disabled':True,
    'no_retraining_or_recalibration':True,
}
first_failure=next((k for k,v in checks.items() if not v),None)
verdict='PASS_V9_0B_FRESH_SAFE_REMOVE_BATCH' if first_failure is None else 'FAIL_OR_PARTIAL_V9_0B_BATCH'
record={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'60R1-v9.0B-fresh-safe-remove-HITL','verdict':verdict,'first_failed_gate':first_failure,'checks':checks,'resolved_pages':len(roster),'passed_pages':len(page_df),'failed_pages':len(fail_df),'expected_page_reference':EXPECTED_REVIEWED_PAGES,'policy_id':policy.get('policy_id'),'compatibility_id':policy.get('compatibility_id'),'notebook52_run':str(N52),'safe_remove_total':int(page_df.safe_remove_pixels.sum()),'m3_total':int(page_df.m3_pixels.sum()),'m5_total':int(page_df.m5_pixels.sum()),'clean_truth_used':False,'safe_protect_enabled':False,'automatic_production_authority':False}
atomic_json(OUT/'notebook60r1_verdict.json',record)
print(json.dumps(record,indent=2))

## 10. Visual review queue

In [ ]:
queue=page_df.copy(); queue['safe_remove_fraction']=queue.safe_remove_pixels/np.maximum(queue.m3_pixels,1); queue=queue.sort_values(['safe_remove_fraction','safe_remove_pixels'],ascending=False); queue.to_csv(TABLES/'visual_review_queue.csv',index=False)
for rank,(_,r) in enumerate(queue.head(MAX_VISUAL_PAGES).iterrows(),1):
    page=r.page; observed=read_gray(PAGES/page/'observed.png'); m3=read_gray(PAGES/page/'m3.png')>0; safe=read_gray(PAGES/page/'safe_remove.png')>0; m5=read_gray(PAGES/page/'m5_remove.png')>0; base=read_gray(PAGES/page/'v7_8_baseline.png'); corrected=read_gray(PAGES/page/'corrected.png')
    overlay=cv2.cvtColor(observed,cv2.COLOR_GRAY2RGB); overlay[m3]=(255,180,0); overlay[safe]=(0,200,0); overlay[m5]=(220,80,40)
    fig,ax=plt.subplots(1,4,figsize=(20,7)); ims=[observed,overlay,base,corrected]; titles=['Observed','M3 orange / SAFE_REMOVE green / M5 red','V7.8 baseline','V8.8 REMOVE-only result']
    for a,im,t in zip(ax,ims,titles): a.imshow(im,cmap='gray' if im.ndim==2 else None); a.set_title(t); a.axis('off')
    fig.suptitle(f'{page} | SAFE_REMOVE={int(r.safe_remove_pixels):,} | M3={int(r.m3_pixels):,} | fraction={r.safe_remove_fraction:.4f}'); plt.tight_layout(); fig.savefig(VISUAL/f'{rank:03d}_{page}.png',dpi=130,bbox_inches='tight'); plt.close(fig)
print('Visual cases:',min(MAX_VISUAL_PAGES,len(queue)))

## 11. Identity, summary and final upload package

In [ ]:
environment={'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0),'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()}
identity={'created_utc':datetime.now(timezone.utc).isoformat(),'run':str(OUT),'version':'60R1-v9.0B-fresh-safe-remove-HITL','notebook52':str(N52),'policy_sha256':sha256_file(POLICY_PATH),'inventory_sha256':sha256_file(INVENTORY_PATH),'replay_sha256':sha256_file(REPLAY_PATH),'schema_sha256':sha256_file(SCHEMA_PATH),'models':[{'path':str(p),'sha256':sha256_file(p)} for p in MODEL_PATHS],'calibrators':[{'path':str(p),'sha256':sha256_file(p)} for p in CALIBRATOR_PATHS],'contract':contract,'reviewed_roster_sha256':sha256_file(TABLES/'hitl_reviewed_roster.csv'),'page_results_sha256':sha256_file(TABLES/'page_results.csv'),'predictions_sha256':sha256_file(TABLES/'all_fresh_safe_remove_predictions.parquet') if (TABLES/'all_fresh_safe_remove_predictions.parquet').exists() else None,'environment':environment}
atomic_json(OUT/'experiment_identity.json',identity)
lines=['NOTEBOOK 60R1 / V9.0B FRESH SAFE_REMOVE HITL OPERATIONALISATION','',f'Verdict: {verdict}',f'Reviewed pages resolved: {len(roster)}',f'Passed pages: {len(page_df)}',f'Failed pages: {len(fail_df)}',f'Frozen policy ID: {policy.get("policy_id")}',f'SAFE_REMOVE pixels: {int(page_df.safe_remove_pixels.sum()):,}',f'M3 pixels: {int(page_df.m3_pixels.sum()):,}',f'M5_REMOVE pixels: {int(page_df.m5_pixels.sum()):,}','','No Notebook 45/46/50 page-output dependency.','No clean truth at inference.','No retraining, refitting, recalibration or threshold movement.','SAFE_PROTECT disabled.','Production authority remains false.']
(OUT/'executive_summary.txt').write_text(chr(10).join(lines))

required=[OUT/'executive_summary.txt',OUT/'notebook60r1_verdict.json',OUT/'experiment_identity.json',AUDIT/'notebook52_source.json',AUDIT/'fresh_inference_contract.json',AUDIT/'resolved_frozen_artifacts.csv',AUDIT/'deterministic_replay.json',TABLES/'hitl_reviewed_roster.csv',TABLES/'page_results.csv',TABLES/'page_failures.csv',TABLES/'visual_review_queue.csv']
if (TABLES/'all_fresh_safe_remove_predictions.parquet').exists(): required.append(TABLES/'all_fresh_safe_remove_predictions.parquet')
for p in required: require(p.exists(),f'Missing upload artifact: {p}')
rows=[]
for src in required+sorted(VISUAL.glob('*.png'))[:MAX_VISUAL_PAGES]:
    cat='visuals' if src.suffix.lower()=='.png' else ('tables' if src.parent==TABLES else 'audit' if src.parent==AUDIT else 'root'); dst=UPLOAD/cat/src.name; dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(src,dst); rows.append({'relative_path':str(dst.relative_to(UPLOAD)),'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})
pd.DataFrame(rows).sort_values('relative_path').to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'README.txt').write_text('Notebook 60R1 fresh SAFE_REMOVE inference on Notebook 07 reviewed HITL masks. Research/shadow evidence only; no automatic production authority.' + chr(10))
zip_path=UPLOAD/'60R1_V9_0B_review_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(UPLOAD.rglob('*')):
        if p.is_file() and p!=zip_path: z.write(p,p.relative_to(UPLOAD))
require(zip_path.exists() and zip_path.stat().st_size>0,'Upload bundle creation failed.')
print(chr(10).join(lines)); print('Upload bundle:',zip_path)